# 中间件

`Starlette`使用中间件来扩展功能和在请求处理流程上进行请求接收和请求处理之间的额外处理过程。其中`Starlette`也提供了不少内置的中间件来完成相应的功能。中间件可以使用`Starlette`类中提供的`add_middleware()`方法来添加到处理流程中。`add_middleware()`方法针对不同的中间件，接受不同的命名参数。

以下就几个常用的中间件进行简单的介绍。

## 使用中间件
Starlette 应用程序类允许您以确保它仍然被异常处理程序包装的方式包含 ASGI 中间件。



In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.httpsredirect import HTTPSRedirectMiddleware
from starlette.middleware.trustedhost import TrustedHostMiddleware

routes = ...

# Ensure that all requests include an 'example.com' or
# '*.example.com' host header, and strictly enforce https-only access.
middleware = [
    Middleware(
        TrustedHostMiddleware,
        allowed_hosts=['example.com', '*.example.com'],
    ),
    Middleware(HTTPSRedirectMiddleware)
]

app = Starlette(routes=routes, middleware=middleware)
# app.add_middleware(...)

每个 Starlette 应用程序默认自动包含两个中间件：

- `ServerErrorMiddleware`- 确保应用程序异常可以返回自定义的 500 页面，或在 DEBUG 模式下显示应用程序回溯。这始终是最外层的中间件层。
- `ExceptionMiddleware`- 添加异常处理程序，以便特定类型的预期异常情况可以与处理程序函数关联。例如，HTTPException(status_code=404)在端点内引发异常最终将呈现自定义 404 页面。

**中间件从上到下进行评估，因此我们示例应用程序的执行流程如下**：

- Middleware
    - ServerErrorMiddleware
    - TrustedHostMiddleware
    - HTTPSRedirectMiddleware
    - ExceptionMiddleware
- Routing
- Endpoint

Starlette 软件包中提供以下中间件实现：

## CORSMiddleware
向传出响应添加适当的CORS 标头，以允许来自浏览器的跨域请求。

`CORSMiddleware` 实现使用的默认参数默认是受限制的，因此您需要明确启用特定的来源、方法或标头，以便允许浏览器在跨域上下文中使用它们。




In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware

routes = ...

middleware = [
    Middleware(CORSMiddleware, allow_origins=['*'])
]

app = Starlette(routes=routes, middleware=middleware)

支持以下论点：

- `allow_origins`- 允许进行跨域请求的来源列表。例如。`['https://example.org', 'https://www.example.org']`您可以使用它['*']来允许任何来源。
- `allow_origin_regex`- 用于匹配应允许发出跨域请求的来源的正则表达式字符串。例如`'https://.*\.example\.org'`。
- `allow_methods`- 跨域请求允许的 HTTP 方法列表。默认为['GET']。您可以使用['*']来允许所有标准方法。
- `allow_headers`- 跨域请求应支持的 HTTP 请求标头列表。默认为[]。您可以使用['*']来允许所有标头。CORS 请求始终允许使用 `Accept`、`Accept-Language`、`Content-Language` 和 `Content-Type` 标头。
- `allow_credentials`- 指示跨域请求应支持 `Cookie`。默认为`False`。此外，`allow_origins`、`allow_methods` 和 `allow_headers` 不能设置为 ['*'] 才能允许凭据，必须显式指定所有这些凭据。
- `expose_headers`- 指示浏览器应访问的任何响应标头。默认为[]。
- `max_age`- 设置浏览器缓存 CORS 响应的最长时间（以秒为单位）。默认为600。

中间件响应两种特定类型的 HTTP 请求...

### CORS 预检请求
这些是带有 `Origin` 和 `Access-Control-Request-Method` 标头的任何 `OPTIONS` 请求。在这种情况下，中间件将拦截传入的请求，并使用适当的 CORS 标头以及 200 或 400 响应进行响应，以供参考。

**简单请求**
任何带有标头的请求`Origin`。在这种情况下，中间件将照常传递请求，但会在响应中包含适当的 `CORS` 标头。

### CORSMiddleware 

在 Starlette 应用程序中使用 `CORSMiddleware` 时，务必确保即使未处理的异常生成的错误响应也应用 CORS 标头。推荐的解决方案是使用 `CORSMiddleware `包装整个 Starlette 应用程序。这种方法可以保证即使 `ServerErrorMiddleware（`或其他外部错误处理中间件）捕获了异常，响应仍将包含正确的`Access-Control-Allow-Origin`标头。

In [ ]:
from starlette.applications import Starlette
from starlette.middleware.cors import CORSMiddleware

import uvicorn

app = Starlette()
app = CORSMiddleware(app=app, allow_origins=["*"])

# ... your routes and middleware configuration ...

if __name__ == '__main__':
    uvicorn.run(
        app,
        host='0.0.0.0',
        port=8000
    )

## SessionMiddleware

`SessionMiddleware`用于提供基于Cookie的会话支持。加载SessionMiddleware后，可以使用request.session来访问保存在会话中的内容。`SessionMiddleware`在`add_middleware()`方法中接受的参数有以下这些。

- `secret_key`，用于加密Session的密钥。
- `session_cookie`，用于保存Session ID的Cookie名称。
- `max_age`，Session最长的存活时间，默认为2周。
- `same_site`，用于配置是否允许跨站发送Session Cookie。
- `https_only`，配置是否仅在HTTPS站上使用Session。
- `domain` - 用于在子域或跨域之间共享 Cookie 的 Cookie 的域。浏览器默认将域设置为设置 Cookie 的主机，但不包括子域（参考）。


In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.sessions import SessionMiddleware

routes = ...

middleware = [
    Middleware(SessionMiddleware, secret_key=..., https_only=True)
]

app = Starlette(routes=routes, middleware=middleware)

## HTTPSRedirectMiddleware
强制所有传入请求必须是 `https` 或 `wss`。对 `http` 或 `ws` 的任何传入请求都将重定向到安全方案。



In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.httpsredirect import HTTPSRedirectMiddleware

routes = ...

middleware = [
    Middleware(HTTPSRedirectMiddleware)
]

app = Starlette(routes=routes, middleware=middleware)

此中间件类没有配置选项。

## TrustedHostMiddleware
`TrustedHostMiddleware`主要用于配置可信网站，用于防止HTTP头攻击。在配置时只接受一个参数：`allowed_hosts`，字符串列表类型，用于列举可信主机。



In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.trustedhost import TrustedHostMiddleware

routes = ...

middleware = [
    Middleware(TrustedHostMiddleware, allowed_hosts=['example.com', '*.example.com'])
]

app = Starlette(routes=routes, middleware=middleware)

- `allowed_hosts`- 允许作为主机名的域名列表。*.example.com支持使用通配符域名（例如，用于匹配子域名）。要允许任何主机名，请使用`allowed_hosts=["*"]`或省略中间件。
- `www_redirect`- 如果设置为 `True`，则对允许主机的非 `www` 版本的请求将被重定向到其对应的 `www` 版本。默认为`True`。

如果传入请求未正确验证，则会发送 400 响应。

## GZipMiddleware

`GZipMiddleware`用于对响应进行压缩，降低响应在网络上的传输时间。在配置时只接受一个参数：`minimum_size`，整型值，用于表示低于此大小（字节数）的响应将不会被压缩，该值默认为500。



In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.gzip import GZipMiddleware


routes = ...

middleware = [
    Middleware(GZipMiddleware, minimum_size=1000, compresslevel=9)
]

app = Starlette(routes=routes, middleware=middleware)

- `minimum_size`- 请勿对小于此最小字节数的响应进行 GZip 压缩。默认为500。
- `compresslevel`- 用于 GZip 压缩。它是一个介于 1 到 9 之间的整数。默认为9。值越低，压缩速度越快，但文件体积越大；值越高，压缩速度越慢，但文件体积越小。

中间件不会对已经设置了 `Content-Encoding` 或 `Content-Type` 设置为 `text/event-stream`（避免压缩服务器发送的事件）的响应进行 GZip 压缩。

## BaseHTTPMiddleware

`BaseHTTPMiddleware`并不是一个可以直接使用的中间件，而是一个中间件的抽象基类。通过继承`BaseHTTPMiddleware`类并实现其中的`async def dispatch(self, request, call_next)`方法可以自定义一个中间件。

`BaseHTTPMiddleware`的`__init__()`构造函数通常需要为以下格式：`def __init__(self, app, **kwargs)`，其中`app`为当前`Starlette`应用的实例，而后面的`**kwargs`命名参数则是可以从`add_middleware()`方法处获取的命名参数。

In [ ]:
class CustomHeaderMiddleware(BaseHTTPMiddleware):
    def __init__(self, app, header_value='Example'):
        super().__init__(app)
        self.header_value = header_value

    async def dispatch(self, request, call_next):
        response = await call_next(request)
        response.headers['Custom'] = self.header_value
        return response


middleware = [
    Middleware(CustomHeaderMiddleware, header_value='Customized')
]

app = Starlette(routes=routes, middleware=middleware)

### 局限性

目前，`BaseHTTPMiddleware` 有一些已知的限制：

使用 `BaseHTTPMiddleware` 将防止 `contextvars.ContextVars` 的更改向上传播。也就是说，如果您在端点中为 `ContextVar` 设置了一个值，然后尝试从中间件中读取该值，您会发现该值与您在端点中设置的值不同（有关此行为的示例，请参阅此测试）。

为了克服这些限制，请使用`pure ASGI middleware`，如下所示。



In [ ]:
import contextvars #作用：为每个线程创建一个变量

import pytest
from starlette.types import ASGIApp, Receive, Scope, Send
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.routing import Mount, Route, WebSocketRoute
from starlette.responses import PlainTextResponse, StreamingResponse

ctxvar: contextvars.ContextVar[str] = contextvars.ContextVar("ctxvar")


class CustomMiddlewareWithoutBaseHTTPMiddleware:
    def __init__(self, app: ASGIApp) -> None:
        self.app = app

    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:
        ctxvar.set("set by middleware")
        await self.app(scope, receive, send)
        assert ctxvar.get() == "set by endpoint"


class CustomMiddlewareUsingBaseHTTPMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        ctxvar.set("set by middleware")
        resp = await call_next(request)
        assert ctxvar.get() == "set by endpoint"
        return resp  # pragma: no cover

@pytest.mark.parametrize(
    "middleware_cls",
    [
        CustomMiddlewareWithoutBaseHTTPMiddleware,
        pytest.param(
            CustomMiddlewareUsingBaseHTTPMiddleware,
            marks=pytest.mark.xfail(
                reason=(
                    "BaseHTTPMiddleware creates a TaskGroup which copies the context"
                    "and erases any changes to it made within the TaskGroup"
                ),
                raises=AssertionError,
            ),
        ),
    ],
)
def test_contextvars(test_client_factory, middleware_cls: type):
    # this has to be an async endpoint because Starlette calls run_in_threadpool
    # on sync endpoints which has it's own set of peculiarities w.r.t propagating
    # contextvars (it propagates them forwards but not backwards)
    async def homepage(request):
        assert ctxvar.get() == "set by middleware"
        ctxvar.set("set by endpoint")
        return PlainTextResponse("Homepage")

    app = Starlette(
        middleware=[Middleware(middleware_cls)], routes=[Route("/", homepage)]
    )

    client = test_client_factory(app)
    response = client.get("/")
    assert response.status_code == 200, response.content

### Pure ASGI Middleware

ASGI 规范使得直接使用 ASGI 接口实现 ASGI 中间件成为可能，就像 ASGI 应用程序链一样，可以调用下一个应用程序。事实上，`Starlette`中的中间件类就是这样实现的。

这种低级方法可提供更强的行为控制能力，并增强跨框架和跨服务器的互操作性。它还克服了 `BaseHTTPMiddleware` 的局限性。

#### 写Pure AsgiMiddleware

创建ASGI中间件的最常见方法是与一类使用。

In [1]:
class ASGIMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        await self.app(scope, receive, send)

上面的中间件是最基本的 ASGI 中间件。它接收一个父 ASGI 应用程序作为其构造函数的参数，并实现一个`async __call__`调用该父应用程序的方法。

一些实现例如`asgi-cors`使用替代样式，使用函数：

In [ ]:
import functools

def asgi_middleware():
    def asgi_decorator(app):

        @functools.wraps(app)
        async def wrapped_app(scope, receive, send):
            await app(scope, receive, send)

        return wrapped_app

    return asgi_decorator

无论如何，ASGI 中间件必须是接受三个参数的可调用函数：`scope`、`receive`、和`send`。

- `scope`是一个保存有关连接信息的字典，其中scope["type"]可能包括：
    - `"http"`：用于 HTTP 请求。
    - `"websocket"`：用于 WebSocket 连接。
    - `"lifespan"`：用于 ASGI 生命周期消息。

`receive`并可`send`用于与 ASGI 服务器交换 ASGI 事件消息——下文将详细介绍。这些消息的类型和内容取决于作用域类型。更多信息请参阅ASGI 规范。


使用纯 ASGI 中间件        
纯 ASGI 中间件可以像任何其他中间件一样使用：


In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware

from .middleware import ASGIMiddleware

routes = ...

middleware = [
    Middleware(ASGIMiddleware),
]

app = Starlette(..., middleware=middleware)

#### 类型注解
注释中间件有两种方法：使用 `Starlette` 本身或`asgiref`。

- 使用 Starlette：适用于最常见的用例。


In [ ]:
from starlette.types import ASGIApp, Message, Scope, Receive, Send


class ASGIMiddleware:
    def __init__(self, app: ASGIApp) -> None:
        self.app = app

    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:
        if scope["type"] != "http":
            return await self.app(scope, receive, send)

        async def send_wrapper(message: Message) -> None:
            # ... Do something
            await send(message)

        await self.app(scope, receive, send_wrapper)

- 使用asgiref: 进行更严格的类型提示。

In [ ]:
from asgiref.typing import ASGI3Application, ASGIReceiveCallable, ASGISendCallable, Scope
from asgiref.typing import ASGIReceiveEvent, ASGISendEvent


class ASGIMiddleware:
    def __init__(self, app: ASGI3Application) -> None:
        self.app = app

    async def __call__(self, scope: Scope, receive: ASGIReceiveCallable, send: ASGISendCallable) -> None:
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        async def send_wrapper(message: ASGISendEvent) -> None:
            # ... Do something
            await send(message)

        return await self.app(scope, receive, send_wrapper)

### 常见模式
#### 仅处理某些请求

ASGI中间件可以根据的内容应用特定的行为`scope`。

例如，为了仅处理 HTTP 请求，请这样写...

In [ ]:
class ASGIMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        ...  # Do something here!

        await self.app(scope, receive, send)

同样，仅支持 WebSocket 的中间件将会进行保护`scope["type"] != "websocket`"。

中间件也可能根据请求方法、URL、标头等采取不同的行动。

#### 重用 Starlette 组件
`Starlette` 提供了几种接受 `ASGI scope`、`receive`和`/`或`send`参数的数据结构，使您可以在更高的抽象级别上工作。这些数据结构包括`Request`、`Headers`、`QueryParams`、`URL`等。

例如，您可以实例化Request以更轻松地检查 HTTP 请求：

In [ ]:
from starlette.requests import Request

class ASGIMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] == "http":
            request = Request(scope)
            ... # Use `request.method`, `request.url`, `request.headers`, etc.

        await self.app(scope, receive, send)

您还可以重复使用响应，它们也是 ASGI 应用程序。

#### 发送热切的回应
检查连接`scope`允许您有条件地调用其他 ASGI 应用。一个用例可能是在不调用该应用的情况下发送响应。

例如，此中间件使用字典根据请求路径执行永久重定向。如果您需要重构路由 `URL` 模式，这可以用于实现对旧版 `URL` 的持续支持。




In [ ]:
from starlette.datastructures import URL
from starlette.responses import RedirectResponse

class RedirectsMiddleware:
    def __init__(self, app, path_mapping: dict):
        self.app = app
        self.path_mapping = path_mapping

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        url = URL(scope=scope)

        if url.path in self.path_mapping:
            url = url.replace(path=self.path_mapping[url.path])
            response = RedirectResponse(url, status_code=301)
            await response(scope, receive, send)
            return

        await self.app(scope, receive, send)

示例用法如下：

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware

routes = ...

redirections = {
    "/v1/resource/": "/v2/resource/",
    # ...
}

middleware = [
    Middleware(RedirectsMiddleware, path_mapping=redirections),
]

app = Starlette(routes=routes, middleware=middleware)

#### 检查或修改请求

可以通过操作`scope`来访问或更改请求信息。有关这种模式的完整示例，请参阅 `Uvicorn` 的 `ProxyHeadersMiddleware`，它可以在前端代理服务时检查和调整`scope`。

此外，对 `receive ASGI` 可调用程序进行封装后，就可以通过操作 `http.request ASGI` 事件消息来访问或修改 `HTTP` 请求正文。

例如，该中间件会计算并记录传入请求正文的大小...

In [ ]:
class LoggedRequestBodySizeMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        body_size = 0

        async def receive_logging_request_body_size():
            nonlocal body_size

            message = await receive()
            assert message["type"] == "http.request"

            body_size += len(message.get("body", b""))

            if not message.get("more_body", False):
                print(f"Size of request body was: {body_size} bytes")

            return message

        await self.app(scope, receive_logging_request_body_size, send)

同样，`WebSocket` 中间件也可以操纵 `websocket.receive ASGI` 事件消息来检查或更改传入的 `WebSocket` 数据。

有关更改 `HTTP` 请求正文的示例，请参阅 `msgpack-asgi`。


#### 检查或修改响应

包装sendASGI 可调用函数可以让你检查或修改底层应用程序发送的 HTTP 响应。为此，请对`http.response.startASGIhttp.response.body`事件消息做出响应。

例如，该中间件添加了一些固定的额外响应标头：

In [ ]:
from starlette.datastructures import MutableHeaders

class ExtraResponseHeadersMiddleware:
    def __init__(self, app, headers):
        self.app = app
        self.headers = headers

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.app(scope, receive, send)

        async def send_with_extra_headers(message):
            if message["type"] == "http.response.start":
                headers = MutableHeaders(scope=message)
                for key, value in self.headers:
                    headers.append(key, value)

            await send(message)

        await self.app(scope, receive, send_with_extra_headers)

示例用法如下：

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware

routes = ...

redirections = {
    "/v1/resource/": "/v2/resource/",
    # ...
}

middleware = [
    Middleware(RedirectsMiddleware, path_mapping=redirections),
]

app = Starlette(routes=routes, middleware=middleware)

#### 检查或修改请求
可以通过操作 `scope` 来访问或更改请求信息。有关这种模式的完整示例，请参阅 `Uvicorn` 的 `ProxyHeadersMiddleware`，它可以在前端代理服务时检查并调整范围。

此外，对 `receive ASGI` 可调用程序进行封装后，就可以通过操作 `http.request ASGI` 事件消息来访问或修改 `HTTP` 请求正文。

例如，该中间件计算并记录传入请求主体的大小……

In [ ]:
class LoggedRequestBodySizeMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        body_size = 0

        async def receive_logging_request_body_size():
            nonlocal body_size

            message = await receive()
            assert message["type"] == "http.request"

            body_size += len(message.get("body", b""))

            if not message.get("more_body", False):
                print(f"Size of request body was: {body_size} bytes")

            return message

        await self.app(scope, receive_logging_request_body_size, send)

同样，`WebSocket` 中间件也可以操纵 `websocket.receive ASG`I 事件消息来检查或更改传入的 `WebSocket` 数据。

有关更改 HTTP 请求正文的示例，请参阅 `msgpack-asgi`。

#### 检查或修改响应

对`send` ASGI 可调用程序进行包装后，就可以检查或修改底层应用程序发送的 HTTP 响应。 为此，请对 `http.response.start` 或 `http.response.body` ASGI 事件消息做出反应。

例如，该中间件添加了一些固定的额外响应标头：

In [ ]:
from starlette.datastructures import MutableHeaders

class ExtraResponseHeadersMiddleware:
    def __init__(self, app, headers):
        self.app = app
        self.headers = headers

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.app(scope, receive, send)

        async def send_with_extra_headers(message):
            if message["type"] == "http.response.start":
                headers = MutableHeaders(scope=message)
                for key, value in self.headers:
                    headers.append(key, value)

            await send(message)

        await self.app(scope, receive, send_with_extra_headers)

另请参阅`asgi-logger`检查 HTTP 响应并记录可配置 HTTP 访问日志行的示例。

同样，WebSocket 中间件可以操纵`websocket.send` ASGI 事件消息来检查或更改传出的 WebSocket 数据。

请注意，如果您更改了响应主体，则需要更新响应`Content-Length`标头以匹配新的响应主体长度。请参阅brotli-asgi完整示例。


#### 将信息传递到端点

如果需要与底层应用或端点共享信息，可以将其存储在`scope`字典中。请注意，这只是一个惯例——例如，`Starlette` 使用它与端点共享路由信息——但它并非 ASGI 规范的一部分。如果这样做，请务必使用不太可能被其他中间件或应用程序使用的键来避免冲突。

例如，当包含下面的中间件时，端点将能够访问`request.scope["asgi_transaction_id"]`。

In [ ]:
import uuid

class TransactionIDMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        scope["asgi_transaction_id"] = uuid.uuid4()
        await self.app(scope, receive, send)

#### 清理和错误处理

您可以将应用程序包装在`try/except/finally`块或上下文管理器中以执行清理操作或进行错误处理。

例如，以下中间件可能会收集指标并处理应用程序异常……

In [ ]:
import time

class MonitoringMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        start = time.time()
        try:
            await self.app(scope, receive, send)
        except Exception as exc:
            ...  # Process the exception
            raise
        finally:
            end = time.time()
            elapsed = end - start
            ...  # Submit `elapsed` as a metric to a monitoring backend

另请参阅`timing-asgi`此模式的完整示例。

## 陷阱
### ASGI 中间件应该是无状态的

由于 ASGI 旨在处理并发请求，因此任何特定于连接的状态都应限定在`__call__`实现范围内。不这样做通常会导致跨请求的变量读写冲突，并且很可能会导致 bug。

例如，如果`X-Mock`响应中存在标头，这将有条件地替换响应主体......


In [ ]:
### 正确写法
from starlette.datastructures import Headers

class MockResponseBodyMiddleware:
    def __init__(self, app, content):
        self.app = app
        self.content = content

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        # A flag that we will turn `True` if the HTTP response
        # has the 'X-Mock' header.
        # ✅: Scoped to this function.
        should_mock = False

        async def maybe_send_with_mock_content(message):
            nonlocal should_mock

            if message["type"] == "http.response.start":
                headers = Headers(raw=message["headers"])
                should_mock = headers.get("X-Mock") == "1"
                await send(message)

            elif message["type"] == "http.response.body":
                if should_mock:
                    message = {"type": "http.response.body", "body": self.content}
                await send(message)

        await self.app(scope, receive, maybe_send_with_mock_content)

In [ ]:
#### 错误写法
from starlette.datastructures import Headers

class MockResponseBodyMiddleware:
    def __init__(self, app, content):
        self.app = app
        self.content = content
        # ❌: This variable would be read and written across requests!
        self.should_mock = False

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        async def maybe_send_with_mock_content(message):
            if message["type"] == "http.response.start":
                headers = Headers(raw=message["headers"])
                self.should_mock = headers.get("X-Mock") == "1"
                await send(message)

            elif message["type"] == "http.response.body":
                if self.should_mock:
                    message = {"type": "http.response.body", "body": self.content}
                await send(message)

        await self.app(scope, receive, maybe_send_with_mock_content)


另请参阅`GZipMiddleware`解决此潜在问题的完整示例实现。

进一步阅读
该文档足以为如何创建 ASGI 中间件奠定良好的基础。

尽管如此，还是有一些关于这个主题的精彩文章：

- [ASGI 简介：异步 Python Web 生态系统的出现](https://florimond.dev/en/posts/2019/08/introduction-to-asgi-async-python-web/)
- [如何编写 ASGI 中间件](https://pgjones.dev/blog/how-to-write-asgi-middleware-2021/)

## 在其他框架中使用中间件
要将 ASGI 中间件包装在其他 ASGI 应用程序周围，您应该使用更通用的包装应用程序实例的模式：




In [ ]:
app = TrustedHostMiddleware(app, allowed_hosts=['example.com'])

您也可以使用 Starlette 应用程序实例执行此操作，但最好使用`middleware=<List of Middleware instances>`样式，因为它将：

- 确保所有东西都包裹在最外层`ServerErrorMiddleware`。
- 保留顶级app实例。

## 将中间件应用于路由组
中间件也可以添加到Mount实例中，这样你就可以将中间件应用于一组路由或子应用程序：

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.gzip import GZipMiddleware
from starlette.routing import Mount, Route


routes = [
    Mount(
        "/",
        routes=[
            Route(
                "/example",
                endpoint=...,
            )
        ],
        middleware=[Middleware(GZipMiddleware)]
    )
]

app = Starlette(routes=routes)

请注意，以这种方式使用的中间件不像应用于应用程序的中间件那样被包装在异常处理中间件中`Starlette`。这通常不是问题，因为它仅适用于检查或修改的中间件`Response`，即使如此，您可能也不想将此逻辑应用于错误响应。如果您确实希望仅在某些路由上将中间件逻辑应用于错误响应，则有以下几种选择：

- 添加一个`ExceptionMiddleware`到`Mount`
- 向您的中间件添加一个`try/except`块并从那里返回错误响应
- 将标记和处理分成两个中间件，一个用于`Mount`标记响应需要处理（例如通过设置`scope["log-response"] = True`），另一个应用于`Starlette`执行繁重工作的应用程序。

`Route`/类`WebSocket`还接受一个`middleware`参数，允许您将中间件应用于单个路由：




In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.gzip import GZipMiddleware
from starlette.routing import Route


routes = [
    Route(
        "/example",
        endpoint=...,
        middleware=[Middleware(GZipMiddleware)]
    )
]

app = Starlette(routes=routes)

您还可以将中间件应用于`Router`类，这允许您将中间件应用于一组路由：

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.gzip import GZipMiddleware
from starlette.routing import Route, Router


routes = [
    Route("/example", endpoint=...),
    Route("/another", endpoint=...),
]

router = Router(routes=routes, middleware=[Middleware(GZipMiddleware)])

## 第三方中间件
### asgi-auth-github
该中间件为任何 ASGI 应用程序添加了身份验证，要求用户使用其 GitHub 帐户（通过OAuth ）登录。访问权限可以限制为特定用户或特定 GitHub 组织或团队的成员。

### asgi-csrf
用于防御 CSRF 攻击的中间件。该中间件实现了双重提交 Cookie 模式，即设置一个 Cookie，然后将其与 csrftoken 隐藏表单字段或x-csrftokenHTTP 标头进行比较。

### Authlib中间件
使用authlib 的 jwt 模块，直接替代 Starlette 会话中间件。

### Bugsnag中间件
用于将异常记录到Bugsnag 的中间件类。

### CSRFMiddleware
用于防御 CSRF 攻击的中间件。该中间件实现了双重提交 Cookie 模式，即设置一个 Cookie，然后将其与x-csrftokenHTTP 标头进行比较。

### 早期数据中间件
用于检测和拒绝TLSv1.3 早期数据请求的中间件和装饰器。

### Prometheus中间件
用于捕获与请求和响应相关的 Prometheus 指标的中间件类，包括正在进行的请求、时间……

### ProxyHeaders中间件
Uvicorn 包含一个中间件类，用于在使用代理服务器时根据X-Forwarded-Proto和X-Forwarded-For标头确定客户端 IP 地址。对于更复杂的代理配置，您可能需要调整此中间件。

### RateLimit中间件
速率限制中间件。正则表达式匹配 URL；规则灵活；高度可定制。非常易于使用。

### 请求ID中间件
用于读取/生成请求 ID 并将其附加到应用程序日志的中间件类。

### Rollbar中间件
用于将异常、错误和日志消息记录到Rollbar 的中间件类。

### StarletteOpentracing
一个中间件类，向OpenTracing.io兼容的跟踪器发出跟踪信息，并可用于分析和监控分布式应用程序。

### 安全Cookies中间件
可定制的中间件，用于向 Starlette 应用程序添加自动 cookie 加密和解密，并为现有的基于 cookie 的中间件提供额外支持。

### 计时中间件
一个中间件类，用于为每个经过的请求发出时间信息（CPU 和实际时间）。其中包含如何将这些时间信息作为 statsd 指标发出的示例。

### WSGIMiddleware
负责将 WSGI 应用程序转换为 ASGI 应用程序的中间件类。